In [ ]:
# --- Imports para los tres mini-demos del Cap. 3 ---------------------
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, accuracy_score


import sklearn, numpy
print('scikit-learn:', sklearn.__version__)  # anota la versión
print('numpy:', numpy.__version__)           # anota la versión

In [ ]:
# --- Cargar California Housing ------------------------------------------
datos = fetch_california_housing()
X = datos.data       # shape: (20640, 8) — las features
y = datos.target     # shape: (20640,)   — precio medio en $100,000


print('Muestras:', X.shape[0])
print('Features:', X.shape[1])
print(f'Precio mínimo: ${round(y.min() * 100_000):,}')
print(f'Precio máximo: ${round(y.max() * 100_000):,}')


# Convertir a DataFrame para inspeccionarlo con más comodidad
df = pd.DataFrame(X, columns=datos.feature_names)
df['precio'] = y
print(df.head(3))

In [ ]:
# --- Regresión: predecir precio de vivienda ----------------------------
rng = np.random.default_rng(42)   # semilla reproducible


# Dividir: 80% entrenamiento, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Escalar features (la regresión es sensible a las escalas)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


# El patrón de scikit-learn: crear → entrenar → predecir
model_reg = LinearRegression()
model_reg.fit(X_train_s, y_train)    # <-- aquí aprende
predictions = model_reg.predict(X_test_s)


# RMSE: error típico de predicción (en unidades de $100,000)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f'RMSE: {rmse:.3f}')   # ~0.72 → error medio de $72,000

In [ ]:
# --- Clasificación: ¿bloque caro o barato? ----------------------------
# Crear etiqueta binaria: 1 = caro, 0 = barato
mediana = np.median(y)
y_clase = (y > mediana).astype(int)   # array de 0s y 1s


X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_clase, test_size=0.2, random_state=42
)


# El clasificador también sigue el mismo patrón
model_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
model_clf.fit(X_tr_c, y_tr_c)        # <-- aquí aprende
pred_clase = model_clf.predict(X_te_c)


acc = accuracy_score(y_te_c, pred_clase)
print(f'Accuracy: {acc:.3f}')   # ~0.80 → 80% de aciertos

In [ ]:
# --- Clustering: segmentar bloques residenciales ----------------------
# El clustering no usa y (no hay etiquetas)
scaler2 = StandardScaler()
X_scaled = scaler2.fit_transform(X)


# K-Means: busca k=4 grupos en los datos
# Usamos k=4 como punto de partida; el Cap. 10 enseña a elegir k
# con el método del codo (elbow method) de forma sistemática.
model_km = KMeans(n_clusters=4, random_state=42, n_init='auto')
model_km.fit(X_scaled)           # <-- aquí aprende
groups = model_km.labels_        # etiqueta asignada a cada muestra


# model_km.inertia_: suma de distancias al centroide de cada grupo.
# Un k mayor siempre baja la inercia; el Cap. 10 explica cómo
# usarla correctamente para elegir k.
print(f'Inercia: {model_km.inertia_:.1f}')


# Ver cuántas muestras cayeron en cada grupo
for g in range(4):
    n = np.sum(groups == g)
    precio_medio = y[groups == g].mean()
    print(f'Grupo {g}: {n:5d} bloques | precio medio: {precio_medio:.2f}')